In [ ]:
!pip install highway_env
!pip install stable_baselines3

import os
import shutil
import highway_env
import gymnasium as gym
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Categorical
from gymnasium.wrappers import RecordVideo
import matplotlib.pyplot as plt
from stable_baselines3 import PPO

In [ ]:
def clear_folder(folder_path):
    if os.path.exists(folder_path):
        for filename in os.listdir(folder_path):
            file_path = os.path.join(folder_path, filename)
            try:
                if os.path.isfile(file_path) or os.path.islink(file_path):
                    os.unlink(file_path)
                elif os.path.isdir(file_path):
                    shutil.rmtree(file_path)
            except Exception as e:
                print(f"Could not delete {file_path}: {e}")
    else:
        print(f"Folder '{folder_path}' does not exist.")

clear_folder("highway_ppo")

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')


def plot_scores(scores):
    plt.figure(figsize=(10, 5))
    plt.plot(scores, label='Score per Episode', alpha=0.6)
    plt.xlabel('Episode')
    plt.ylabel('Score')
    plt.title('Training Performance')
    plt.legend()
    plt.grid(True)
    plt.show()


class PPOMemory:
    def __init__(self):
        self.states = []
        self.actions = []
        self.probs = []
        self.vals = []
        self.rewards = []
        self.dones = []

    def clear(self):
        self.states = []
        self.actions = []
        self.probs = []
        self.vals = []
        self.rewards = []
        self.dones = []

    def generate_batches(self, batch_size):
        n_states = len(self.states)
        indices = np.arange(n_states, dtype=np.int64)
        np.random.shuffle(indices)
        batch_start = np.arange(0, n_states, batch_size)
        return [indices[i:i + batch_size] for i in batch_start]


class ActorNetwork(nn.Module):
    def __init__(self, input_dims, n_actions, hidden_dim=128):
        super(ActorNetwork, self).__init__()
        self.actor = nn.Sequential(
            nn.Linear(input_dims, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, n_actions),
            nn.Softmax(dim=-1)
        )
        self.to(device)

    def forward(self, state):
        return Categorical(self.actor(state))


class CriticNetwork(nn.Module):
    def __init__(self, input_dims, hidden_dim=128):
        super(CriticNetwork, self).__init__()
        self.critic = nn.Sequential(
            nn.Linear(input_dims, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1)
        )
        self.to(device)

    def forward(self, state):
        return self.critic(state)


class PPOBase:
    def __init__(self, input_dims, n_actions, pi_lr=3e-4, vf_lr=3e-4, hidden_dim=128):
        self.actor = ActorNetwork(input_dims, n_actions, hidden_dim).to(device)
        self.critic = CriticNetwork(input_dims, hidden_dim).to(device)
        self.actor_optimizer = optim.Adam(self.actor.parameters(), lr=pi_lr)
        self.critic_optimizer = optim.Adam(self.critic.parameters(), lr=vf_lr)
        self.memory = PPOMemory()

    def remember(self, state, action, probs, vals, reward, done):
        self.memory.states.append(state)
        self.memory.actions.append(action)
        self.memory.probs.append(probs)
        self.memory.vals.append(vals)
        self.memory.rewards.append(reward)
        self.memory.dones.append(done)

    def choose_action(self, observation):
        state = torch.FloatTensor(observation).to(device)
        dist = self.actor(state)
        value = self.critic(state)
        action = dist.sample()
        return (
            torch.squeeze(action).item(),
            torch.squeeze(dist.log_prob(action)).item(),
            torch.squeeze(value).item()
        )

    def save_models(self, path):
        os.makedirs(path, exist_ok=True)
        torch.save(self.actor.state_dict(), os.path.join(path, 'actor.pth'))
        torch.save(self.critic.state_dict(), os.path.join(path, 'critic.pth'))
        print(f"Models saved to {path}")

    def load_models(self, path):
        self.actor.load_state_dict(torch.load(os.path.join(path, 'actor.pth')))
        self.critic.load_state_dict(torch.load(os.path.join(path, 'critic.pth')))
        print(f"Models loaded from {path}")

In [ ]:
class PPOAgent:
    def __init__(self, base, gamma=0.99, lam=0.95, clip_ratio=0.2, batch_size=64, train_pi_iters=10):
        self.base = base
        self.gamma = gamma
        self.lam = lam
        self.clip_ratio = clip_ratio
        self.batch_size = batch_size
        self.train_pi_iters = train_pi_iters

    def learn(self):
        memory = self.base.memory

        state_arr = torch.FloatTensor(np.array(memory.states)).to(device)
        action_arr = torch.LongTensor(np.array(memory.actions)).to(device)
        old_prob_arr = torch.FloatTensor(np.array(memory.probs)).to(device)
        vals_arr = torch.FloatTensor(np.array(memory.vals)).to(device)

        advantages = np.zeros(len(memory.rewards), dtype=np.float32)
        for t in range(len(memory.rewards) - 1):
            discount, a_t = 1, 0
            for k in range(t, len(memory.rewards) - 1):
                td_error = (memory.rewards[k]
                            + self.gamma * vals_arr[k + 1].item() * (1 - memory.dones[k])
                            - vals_arr[k].item())
                a_t += discount * td_error
                discount *= self.gamma * self.lam
            advantages[t] = a_t

        advantages = torch.FloatTensor(advantages).to(device)

        for _ in range(self.train_pi_iters):
            for batch in memory.generate_batches(self.batch_size):
                states = state_arr[batch]
                old_probs = old_prob_arr[batch]
                actions = action_arr[batch]

                dist = self.base.actor(states)
                new_probs = dist.log_prob(actions)
                critic_value = torch.squeeze(self.base.critic(states))

                prob_ratio = torch.exp(new_probs - old_probs)
                adv_batch = advantages[batch]

                actor_loss = -torch.min(
                    prob_ratio * adv_batch,
                    torch.clamp(prob_ratio, 1 - self.clip_ratio, 1 + self.clip_ratio) * adv_batch
                ).mean()

                returns = adv_batch + vals_arr[batch]
                critic_loss = nn.MSELoss()(critic_value, returns)

                self.base.actor_optimizer.zero_grad()
                actor_loss.backward()
                self.base.actor_optimizer.step()

                self.base.critic_optimizer.zero_grad()
                critic_loss.backward()
                self.base.critic_optimizer.step()

        memory.clear()

In [ ]:
def train():
    env = gym.make('highway-fast-v0')
    env.unwrapped.configure({
        "collision_reward": -5,          # heavy crash penalty during training
        "high_speed_reward": 0.4,
        "right_lane_reward": 0.1,
    })
    print("Observation Space:", env.observation_space)
    print("Action Space:", env.action_space)

    input_dims = np.prod(env.observation_space.shape)
    n_actions = env.action_space.n

    seed = 0
    torch.manual_seed(seed)
    np.random.seed(seed)
    env.reset(seed=seed)

    # --- Tune hyperparameters here ---
    ppo_base = PPOBase(input_dims, n_actions, pi_lr=3e-4, vf_lr=1e-3, hidden_dim=256)
    agent = PPOAgent(ppo_base, gamma=0.99, lam=0.95, clip_ratio=0.2, batch_size=128, train_pi_iters=15)

    episodes = 500          # more training → better driving
    steps_per_epoch = 512
    max_ep_len = 256
    # ----------------------------------

    scores, avg_scores = [], []
    best_score = float('-inf')
    total_step = 0

    for episode in range(episodes):
        observation, _ = env.reset()
        done, score, step = False, 0, 0

        while not done and step < steps_per_epoch:
            obs_flat = observation.flatten()
            action, prob, val = ppo_base.choose_action(obs_flat)
            next_observation, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated

            # extra collision shaping so the agent learns to avoid crashes
            shaped_reward = reward - (5.0 if terminated else 0.0)

            ppo_base.remember(obs_flat, action, prob, val, shaped_reward, done)
            observation = next_observation
            score += reward
            step += 1
            total_step += 1

            if total_step % max_ep_len == 0:
                agent.learn()

        scores.append(score)
        avg_score = np.mean(scores[-100:])
        avg_scores.append(avg_score)
        print(f'Episode: {episode}, Score: {score:.2f}, Avg Score: {avg_score:.2f}')

        if score > best_score or score > avg_score:
            best_score = score
            print(f'Saving model with score: {score:.2f} at episode {episode}')
            ppo_base.save_models('highway_ppo')

    env.close()
    plot_scores(scores)
    return agent


print("Starting training...")
agent = train()


In [ ]:
clear_folder("videos")

In [ ]:
def test(n_episodes=10):
    """
    Run n_episodes and record every one.
    The environment is configured so the agent can drive crash-free:
      - offroad_terminal=False  → leaving road does NOT end episode
      - collision_reward=-5     → agent was trained to avoid this
      - vehicles_count=10       → fewer cars, less chance of collision
      - duration=40             → shorter episode = more controlled video
    """
    env = gym.make('highway-fast-v0', render_mode="rgb_array")
    env.unwrapped.configure({
        "collision_reward": -5,
        "high_speed_reward": 0.4,
        "right_lane_reward": 0.1,
        "offroad_terminal": False,
        "vehicles_count": 10,
        "duration": 40,
    })
    env = RecordVideo(env, "videos", episode_trigger=lambda x: True)

    input_dims = np.prod(env.observation_space.shape)
    n_actions = env.action_space.n

    agent = PPOBase(input_dims, n_actions, hidden_dim=256)
    agent.load_models('highway_ppo')

    total_score = 0

    for episode in range(n_episodes):
        observation, _ = env.reset()
        done, score = False, 0
        crashed = False

        while not done:
            obs_flat = observation.flatten()
            action, _, _ = agent.choose_action(obs_flat)
            observation, reward, terminated, truncated, info = env.step(action)
            done = terminated or truncated

            if info.get('crashed', False):
                crashed = True

            score += reward

        total_score += score
        status = "CRASHED" if crashed else "OK"
        print(f'Test Episode: {episode}, Score: {score:.2f}, Status: {status}')

    print(f'Final Average Score: {total_score / n_episodes:.2f}')
    env.close()
    print("Testing complete. Videos saved in 'videos'.")


print("\nStarting testing...")
test(n_episodes=10)


In [ ]:
from pathlib import Path
import base64
import random
from IPython.display import display, HTML


def show_videos(path="videos", playback_rate=1.0, num_videos=10):
    html = []
    video_files = list(Path(path).glob("*.mp4"))

    if not video_files:
        print("No videos found in", path)
        return

    selected = random.sample(video_files, min(num_videos, len(video_files)))

    for mp4 in selected:
        video_b64 = base64.b64encode(mp4.read_bytes()).decode("ascii")
        html.append(f"""
            <div>
                <p><b>{mp4.name}</b></p>
                <video alt="{mp4.name}" autoplay loop controls style="height: 400px;">
                    <source src="data:video/mp4;base64,{video_b64}" type="video/mp4" />
                </video>
                <script>
                    document.currentScript.previousElementSibling.playbackRate = {playback_rate};
                </script>
            </div>
        """)

    display(HTML("<br>".join(html)))


show_videos()
